# VLE with Activity Coefficients

Computing vapor-liquid equilibrium (VLE) for the ethanol-water system using the NRTL activity coefficient model and Antoine vapor pressure correlation.

## Setup

Import the NRTL activity coefficient model and Antoine vapor pressure correlation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathsim_chem.thermodynamics import NRTL, Antoine

## Pure Component Vapor Pressures

First we set up Antoine correlations for both components. The modified Raoult's law gives:

$$y_i P = x_i \gamma_i P_i^{\text{sat}}(T)$$

At a fixed pressure, this determines the bubble-point temperature and vapor composition.

In [ ]:
# Antoine coefficients (ln form, T in K, P in Pa)
# Ethanol
antoine_ethanol = Antoine(a0=23.5807, a1=3673.81, a2=-46.681)
# Water
antoine_water = Antoine(a0=23.2256, a1=3835.18, a2=-45.343)

def psat(block, T):
    """Evaluate an Antoine block at temperature T."""
    block.inputs[0] = T
    block.update(None)
    return block.outputs[0]

# Check normal boiling points
print(f"Ethanol P_sat at 78.37 °C: {psat(antoine_ethanol, 351.52):.0f} Pa")
print(f"Water P_sat at 100 °C:     {psat(antoine_water, 373.15):.0f} Pa")

## NRTL Activity Coefficients

The NRTL model computes activity coefficients $\gamma_1, \gamma_2$ as a function of temperature and (fixed) composition. These quantify the deviation from ideal mixing.

In [ ]:
def get_gammas(x1, T):
    """Compute NRTL activity coefficients for ethanol(1)-water(2)."""
    nrtl = NRTL(
        x=[x1, 1.0 - x1],
        a=[[0, -0.801], [3.458, 0]],
        c=[[0, 0.3], [0.3, 0]],
    )
    nrtl.inputs[0] = T
    nrtl.update(None)
    return nrtl.outputs[0], nrtl.outputs[1]

# Activity coefficients at 50/50 mixture, 350 K
g1, g2 = get_gammas(0.5, 350)
print(f"gamma_ethanol = {g1:.4f}")
print(f"gamma_water   = {g2:.4f}")

## Activity Coefficient vs Composition

Plot how the activity coefficients vary with liquid composition at a fixed temperature.

In [ ]:
x1_range = np.linspace(0.01, 0.99, 50)
T_fixed = 351.0  # K, near ethanol boiling point

gammas_1 = []
gammas_2 = []
for x1 in x1_range:
    g1, g2 = get_gammas(x1, T_fixed)
    gammas_1.append(g1)
    gammas_2.append(g2)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x1_range, gammas_1, label=r"$\gamma_{\mathrm{ethanol}}$")
ax.plot(x1_range, gammas_2, label=r"$\gamma_{\mathrm{water}}$")
ax.axhline(1.0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel(r"$x_{\mathrm{ethanol}}$")
ax.set_ylabel(r"$\gamma_i$")
ax.set_title(f"NRTL Activity Coefficients at T = {T_fixed} K")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Both activity coefficients approach 1 as the mixture approaches pure component, as expected. The large values at infinite dilution (small $x_i$) reflect the strong non-ideal interactions in this system.

## T-x-y Diagram

Construct the bubble-point T-x-y diagram at atmospheric pressure. For each liquid composition $x_1$, we solve for the temperature $T$ where:

$$\sum_i x_i \gamma_i(T) P_i^{\text{sat}}(T) = P$$

and the vapor composition follows from:

$$y_i = \frac{x_i \gamma_i P_i^{\text{sat}}}{P}$$

In [ ]:
from scipy.optimize import brentq

P_total = 101325  # 1 atm in Pa

def bubble_pressure(x1, T):
    """Compute sum(x_i * gamma_i * P_sat_i) - P for bubble point."""
    x2 = 1.0 - x1
    g1, g2 = get_gammas(x1, T)
    P1 = psat(antoine_ethanol, T)
    P2 = psat(antoine_water, T)
    return x1 * g1 * P1 + x2 * g2 * P2 - P_total

x1_points = np.linspace(0.01, 0.99, 40)
T_bubble = []
y1_values = []

for x1 in x1_points:
    # Solve for bubble-point T between 340 K and 380 K
    T_bp = brentq(lambda T: bubble_pressure(x1, T), 340, 380)
    T_bubble.append(T_bp)
    
    # Vapor composition
    g1, g2 = get_gammas(x1, T_bp)
    P1 = psat(antoine_ethanol, T_bp)
    y1 = x1 * g1 * P1 / P_total
    y1_values.append(y1)

T_bubble = np.array(T_bubble)
y1_values = np.array(y1_values)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x1_points, T_bubble - 273.15, label="Bubble line (liquid)")
ax.plot(y1_values, T_bubble - 273.15, label="Dew line (vapor)")
ax.plot([0, 1], [0, 1], "k--", alpha=0.2, label="y = x")
ax.set_xlabel(r"$x_{\mathrm{ethanol}}, \ y_{\mathrm{ethanol}}$")
ax.set_ylabel("Temperature [°C]")
ax.set_title("Ethanol-Water T-x-y Diagram at 1 atm")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The diagram shows the characteristic minimum-boiling azeotrope of the ethanol-water system near $x_{\mathrm{ethanol}} \approx 0.9$, which is a well-known feature of this system that prevents simple distillation from producing pure ethanol.